# Notebook 06: Đánh giá Tự động Toàn diện Hệ thống RAG (RAG Evaluation)

Notebook này thực hiện đánh giá định lượng (Quantitative Evaluation) hệ thống RAG tiếng Việt trên tập dữ liệu chuẩn `data/eval/testset.jsonl`:

1. **Nạp & Phân tích Bộ câu hỏi Benchmark** (`testset.jsonl`)
2. **Đánh giá tầng Retrieval (Tìm kiếm)**: `Hit Rate @ 1, 3, 5`, `MRR` (Mean Reciprocal Rank), `Context Recall`
3. **Đánh giá tầng Generation (Sinh câu trả lời)**: `Token F1-Score`, `Precision`, `Recall`, `Exact Match`
4. **Kiểm tra Chống Ảo Giác (Anti-Hallucination Rate)**: Đo khả năng từ chối trả lời chính xác đối với các câu hỏi ngoài phạm vi tài liệu (`is_answerable = False`)
5. **Phân tích theo Loại câu hỏi** (`direct`, `reasoning`, `unanswerable`)
6. **So sánh thử nghiệm**: Có Reranker vs Không có Reranker

## 1. Thiết lập Môi trường & Khởi tạo

In [6]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# Thiết lập đường dẫn thư mục gốc dự án
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import get_config
from src.evaluation.dataset import EvaluationDataset
from src.evaluation.evaluator import RAGEvaluator
from src.pipeline.rag_pipeline import RAGPipeline

cfg = get_config(reload=True)
print(f"✅ Dự án: {root}")
print(f"⚙️  LLM Provider: {cfg.llm.default_provider} ({cfg.llm.groq.model_name})")

✅ Dự án: E:\Personal Project\vietnamese-rag
⚙️  LLM Provider: groq (openai/gpt-oss-20b)


## 2. Nạp và Khảo sát Tập Dữ liệu Testset (`testset.jsonl`)

In [2]:
testset_path = root / "data" / "eval" / "testset.jsonl"
dataset = EvaluationDataset.load_jsonl(testset_path)
df_test = dataset.to_pandas()

print(f"Tổng số câu hỏi đánh giá: {len(dataset)}")
display(df_test["question_type"].value_counts().to_frame(name="Số lượng câu"))
display(df_test.head(3)[["id", "query", "ground_truth", "question_type", "is_answerable"]])

2026-10-14 22:01:28 | INFO    | eval_dataset:load_jsonl:80 - Đã nạp thành công 50 mẫu đánh giá từ 'testset.jsonl'
Tổng số câu hỏi đánh giá: 50


,Số lượng câu
question_type,
direct,30
reasoning,10
unanswerable,10


,id,query,ground_truth,question_type,is_answerable
0,q001,Đề tài so sánh hiệu năng của hai mô hình nào t...,DenseNet-201 và EfficientNetB0.,direct,True
1,q002,Bài toán phân loại ảnh X-quang nha khoa trong ...,"5 lớp: Cavity, Fillings, Impacted Tooth, Impla...",direct,True
2,q003,Tập dữ liệu được sử dụng trong đồ án được lấy ...,Từ nền tảng Kaggle.,direct,True


## 3. Khởi tạo Pipeline & Thực thi Đánh giá Toàn diện

In [3]:
pipeline = RAGPipeline()
evaluator = RAGEvaluator(pipeline=pipeline)

2026-10-14 22:01:51 | INFO    | embedder:__init__:39 - Đang tải Embedding model: BAAI/bge-m3 trên thiết bị: cpu


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-10-14 22:02:00 | INFO    | embedder:__init__:49 - Tải model BAAI/bge-m3 thành công. Kích thước vector: 1024
2026-10-14 22:02:00 | INFO    | groq_llm:__init__:34 - Khởi tạo Groq LLM với mô hình: openai/gpt-oss-120b
2026-10-14 22:02:00 | INFO    | prompt_builder:_load_template:47 - Đã nạp prompt template từ: qa_prompt.yaml
2026-10-14 22:02:00 | INFO    | rag_evaluator:__init__:112 - RAGEvaluator sử dụng template: eval_prompt.yaml
2026-10-14 22:02:00 | INFO    | prompt_builder:_load_template:47 - Đã nạp prompt template từ: eval_prompt.yaml
2026-10-14 22:02:01 | INFO    | llm_judge:__init__:63 - Đã khởi tạo LLM Judge với model: openai/gpt-oss-120b


In [3]:
q025 = "Vì sao K-Means được chọn thay vì HAC cho bài toán phân cụm khách hàng Fitness?"
result1 = pipeline.query(q025, top_k=6, top_n=3) 

2026-10-14 15:49:50 | INFO    | rag_pipeline:query:45 - Bắt đầu chu trình RAG Pipeline cho câu hỏi: 'Vì sao K-Means được chọn thay vì HAC cho bài toán phân cụm khách hàng Fitness?'
2026-10-14 15:49:50 | INFO    | retrieval_pipeline:retrieve:59 - Retrieval Pipeline cho query: 'Vì sao K-Means được chọn thay vì HAC cho bài toán phân cụm khách hàng Fitness?' (hybrid=False, rerank=True, top_k=6, top_n=3)
2026-10-14 15:49:50 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Vì sao K-Means được chọn thay vì HAC cho bài toán phân cụm khách hàng Fitness?' (top_k=6)


The `sentences` argument was renamed and is now deprecated. Please use `inputs` instead.


2026-10-14 15:49:50 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 6 chunks tương đồng từ Qdrant.
2026-10-14 15:49:50 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 6 chunks với câu hỏi: 'Vì sao K-Means được chọn thay vì HAC cho bài toán phân cụm khách hàng Fitness?' (top_n=3)
2026-10-14 15:49:57 | INFO    | reranker:rerank:85 - Hoàn tất Reranking. Lọc lại top 3 chunks chất lượng nhất.
2026-10-14 15:49:57 | INFO    | generator:generate_answer:47 - Đang gửi prompt đến LLM để sinh câu trả lời cho query: 'Vì sao K-Means được chọn thay vì HAC cho bài toán phân cụm khách hàng Fitness?'
2026-10-14 15:49:59 | INFO    | rag_pipeline:query:75 - Hoàn tất xử lý RAG Pipeline trong 9.58s


In [4]:
# # Thực hiện đánh giá (Bật đầy đủ Hybrid + Reranker)
result = evaluator.evaluate(
    dataset=dataset,
    # top_k=6,
    # top_n=4,
    use_rerank=True,
    use_hybrid=False,
)

result.print_summary()

2026-10-14 22:02:05 | INFO    | rag_evaluator:evaluate:151 - Bắt đầu đánh giá RAG trên 50 câu hỏi...


Evaluating RAG:   0%|          | 0/50 [00:00<?, ?query/s]

2026-10-14 22:02:05 | INFO    | rag_pipeline:query:45 - Bắt đầu chu trình RAG Pipeline cho câu hỏi: 'Đề tài so sánh hiệu năng của hai mô hình nào trong bài toán phân loại bệnh lý trên răng qua ảnh X-quang?'
2026-10-14 22:02:05 | INFO    | retrieval_pipeline:retrieve:107 - Retrieval Pipeline | hybrid=False | rerank=True | candidate_k=10 | final_n=5
2026-10-14 22:02:05 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Đề tài so sánh hiệu năng của hai mô hình nào trong bài toán phân loại bệnh lý trên răng qua ảnh X-quang?' (top_k=10)


The `sentences` argument was renamed and is now deprecated. Please use `inputs` instead.


2026-10-14 22:02:06 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-14 22:02:06 | INFO    | retrieval_pipeline:retrieve:128 - Candidate Retrieval tra ve 10 chunks.
2026-10-14 22:02:06 | WARNING | retrieval_pipeline:retrieve:157 - Reranker không khả dụng; Giữ các chunk từ retrieval ban đầu.
2026-10-14 22:02:06 | INFO    | retrieval_pipeline:retrieve:168 - Pipeline hoan tat: 5 chunks cuoi cung.
2026-10-14 22:02:06 | INFO    | generator:generate_answer:47 - Đang gửi prompt đến LLM để sinh câu trả lời cho query: 'Đề tài so sánh hiệu năng của hai mô hình nào trong bài toán phân loại bệnh lý trên răng qua ảnh X-quang?'
2026-10-14 22:02:08 | INFO    | rag_pipeline:query:75 - Hoàn tất xử lý RAG Pipeline trong 2.93s


...

2026-10-14 22:15:59 | INFO    | rag_pipeline:query:45 - Bắt đầu chu trình RAG Pipeline cho câu hỏi: 'Dựa vào bảng 1 mô tả cột trong tập dữ liệu Fitness ngày sinh trung bình của hội viên là bao nhiêu ?'
2026-10-14 22:15:59 | INFO    | retrieval_pipeline:retrieve:107 - Retrieval Pipeline | hybrid=False | rerank=True | candidate_k=10 | final_n=5
2026-10-14 22:15:59 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Dựa vào bảng 1 mô tả cột trong tập dữ liệu Fitness ngày sinh trung bình của hội viên là bao nhiêu ?' (top_k=10)
2026-10-14 22:16:00 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-14 22:16:00 | INFO    | retrieval_pipeline:retrieve:128 - Candidate Retrieval tra ve 10 chunks.
2026-10-14 22:16:00 | WARNING | retrieval_pipeline:retrieve:157 - Reranker không khả dụng; Giữ các chunk từ retrieval ban đầu.
2026-10-14 22:16:00 | INFO    | retrieval_pipeline:retrieve:168 - Pipeline hoan tat: 5 chunks cuoi cung.

Evaluating RAG:  98%|█████████▊| 49/50 [14:11<00:18, 18.82s/query]

2026-10-14 22:16:17 | INFO    | rag_pipeline:query:45 - Bắt đầu chu trình RAG Pipeline cho câu hỏi: 'Trong tập dữ liệu Customer Shopping Behavior, có cột nào ghi thông tin về số điện thoại của khách hàng không?'
2026-10-14 22:16:17 | INFO    | retrieval_pipeline:retrieve:107 - Retrieval Pipeline | hybrid=False | rerank=True | candidate_k=10 | final_n=5
2026-10-14 22:16:17 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Trong tập dữ liệu Customer Shopping Behavior, có cột nào ghi thông tin về số điện thoại của khách hàng không?' (top_k=10)
2026-10-14 22:16:17 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-14 22:16:17 | INFO    | retrieval_pipeline:retrieve:128 - Candidate Retrieval tra ve 10 chunks.
2026-10-14 22:16:17 | WARNING | retrieval_pipeline:retrieve:157 - Reranker không khả dụng; Giữ các chunk từ retrieval ban đầu.
2026-10-14 22:16:17 | INFO    | retrieval_pipeline:retrieve:168 - Pipeline hoan tat:

Evaluating RAG: 100%|██████████| 50/50 [14:29<00:00, 17.39s/query]


📊 BÁO CÁO ĐÁNH GIÁ CHẤT LƯỢNG HỆ THỐNG RAG

📌 RETRIEVAL_PERFORMANCE:
  • Total Answerable Queries          : 40
  • Hit Rate @ 1                      : 0.6000 (60.00%)
  • Hit Rate @ 3                      : 0.8250 (82.50%)
  • Hit Rate @ 5                      : 0.9000 (90.00%)
  • MRR (Mean Reciprocal Rank)        : 0.7133 (71.33%)
  • Context Recall                    : 0.8438 (84.38%)

📌 GENERATION_PERFORMANCE:
  • Mean Semantic Similarity (Cosine) : 0.7104 (71.04%)
  • Mean Answer Relevancy (LLM Judge) : 0.8360 (83.60%)
  • Mean Token F1-Score               : 0.3783 (37.83%)
  • Mean Token Precision              : 0.2968 (29.68%)
  • Mean Token Recall                 : 0.6474 (64.74%)
  • Exact Match Rate                  : 0.0500 (5.00%)

📌 ANTI_HALLUCINATION:
  • Total Unanswerable Queries        : 10
  • Correct Rejection Rate (Anti-Hallucination): 1.0000 (100.00%)
  • Hallucination Rate                : 0.0000 (0.00%)

📌 SYSTEM_LATENCY:
  • Total Evaluated Queries           :

## 4. Bảng Kết quả Chi tiết Từng Câu hỏi

In [5]:
df_results = result.to_dataframe()
display(df_results[["id", "query", "hit@1", "hit@3", "mrr", "token_f1", "latency_seconds"]].head(10))

# Lưu kết quả ra file
result.save_json(root / "data" / "eval" / "evaluation_results_no_rerank_hybrid.json")
result.save_csv(root / "data" / "eval" / "evaluation_details_no_rerank_hybrid.csv")
print("✅ Đã lưu kết quả đánh giá vào data/eval/")

,id,query,hit@1,hit@3,mrr,token_f1,latency_seconds
0,q001,Đề tài so sánh hiệu năng của hai mô hình nào t...,1.0,1.0,1.0,0.235294,2.926712
1,q002,Bài toán phân loại ảnh X-quang nha khoa trong ...,1.0,1.0,1.0,0.432432,1.143789
2,q003,Tập dữ liệu được sử dụng trong đồ án được lấy ...,1.0,1.0,1.0,0.380952,2.516855
3,q004,"Số lượng ảnh trong tập train, valid và test củ...",1.0,1.0,1.0,0.888889,1.583305
4,q005,Kiến trúc DenseNet201 được chia thành bao nhiê...,0.0,1.0,0.5,0.500000,10.099216
5,q006,Growth Rate (k) thường dùng trong DenseNet201 ...,1.0,1.0,1.0,0.266667,13.527914
6,q007,Mô hình nào đạt độ chính xác trên tập test cao...,1.0,1.0,1.0,0.093750,14.064105
7,q008,Mô hình DenseNet201 đạt kết quả tốt nhất tại e...,1.0,1.0,1.0,0.000000,12.442998
8,q009,Phương pháp tiền xử lý ảnh đơn lẻ nào có tỷ lệ...,1.0,1.0,1.0,0.700000,11.634785
9,q010,Vì sao nhóm nghiên cứu chọn DenseNet201 làm mô...,1.0,1.0,1.0,0.446429,2.786612


2026-10-14 22:17:32 | INFO    | rag_evaluator:save_json:77 - Đã lưu kết quả đánh giá JSON tại: E:\Personal Project\vietnamese-rag\data\eval\evaluation_results_no_rerank_hybrid.json
2026-10-14 22:17:32 | INFO    | rag_evaluator:save_csv:85 - Đã lưu bảng chi tiết CSV tại: E:\Personal Project\vietnamese-rag\data\eval\evaluation_details_no_rerank_hybrid.csv
✅ Đã lưu kết quả đánh giá vào data/eval/


## 5. Thử nghiệm So sánh: Có Reranker vs Không có Reranker

In [ ]:
# Đánh giá khi TẮT Reranker (chỉ dùng Dense Search)
result_no_rerank = evaluator.evaluate(
    dataset=dataset,
    top_k=3, # Lấy thẳng top 3 từ Vector DB
    use_rerank=False,
)

compare_df = pd.DataFrame([
    {
        "Cấu hình": "Có Reranker (BGE-M3 + Rerank Top 3)",
        "Hit@1": result.summary["retrieval_performance"]["Hit Rate @ 1"],
        "Hit@3": result.summary["retrieval_performance"]["Hit Rate @ 3"],
        "MRR": result.summary["retrieval_performance"]["MRR (Mean Reciprocal Rank)"],
        "Context Recall": result.summary["retrieval_performance"]["Context Recall"],
        "Avg Latency (s)": result.summary["system_latency"]["Average Latency (seconds)"],
    },
    {
        "Cấu hình": "Không Reranker (Chỉ Vector Search Top 3)",
        "Hit@1": result_no_rerank.summary["retrieval_performance"]["Hit Rate @ 1"],
        "Hit@3": result_no_rerank.summary["retrieval_performance"]["Hit Rate @ 3"],
        "MRR": result_no_rerank.summary["retrieval_performance"]["MRR (Mean Reciprocal Rank)"],
        "Context Recall": result_no_rerank.summary["retrieval_performance"]["Context Recall"],
        "Avg Latency (s)": result_no_rerank.summary["system_latency"]["Average Latency (seconds)"],
    }
])

display(compare_df)

In [ ]:
from IPython.display import display, Markdown

refused_ids = [ "q032", "q035", "q037", "q042", "q044"]


def _get(obj, key, default=None):
    """Lấy giá trị từ dict hoặc object."""
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


# Lấy danh sách details từ result (hỗ trợ cả 2 dạng)
details = _get(result, "details", []) or []
# Build dict id -> detail để tra cứu nhanh
detail_map = {_get(d, "id"): d for d in details}


for sample in dataset:
    if sample.id not in refused_ids:
        continue

    chunks = pipeline.retrieval_pipeline.retrieve(
        query=sample.query,
        top_k=10,
        top_n=3,
        use_rerank=True,
        use_hybrid=True,
    )

    # Lấy model answer từ detail tương ứng
    detail = detail_map.get(sample.id)
    model_answer = _get(detail, "generated_answer", "<không có>") if detail else "<không có>"

    md = [f"##  {sample.id}: {sample.query}\n"]
    md.append(f"**Ground truth:** `{sample.ground_truth}`\n")
    md.append(f"** Model trả lời:** `{model_answer}`\n")
    md.append("---\n")

    for i, c in enumerate(chunks, 1):
        # Lấy metadata và text
        if isinstance(c, dict):
            meta = c.get("metadata", {}) or {}
            text = c.get("text", "") or meta.get("text", "")
        else:
            meta = getattr(c, "metadata", {}) or {}
            if not isinstance(meta, dict):
                meta = getattr(meta, "__dict__", {})
            text = (getattr(c, "text", "") or
                    getattr(c, "content", "") or
                    meta.get("text", ""))

        page = meta.get("page_number", "?")
        md.append(f"### 📄 Chunk {i} | page={page}")
        md.append(f"```\n{str(text).strip()}\n```\n")

    display(Markdown("\n".join(md)))

2026-10-11 22:29:50 | INFO    | retrieval_pipeline:retrieve:59 - Retrieval Pipeline cho query: 'Số lượng ảnh lớp Implant trong tập test của bộ dữ liệu là bao nhiêu?' (hybrid=True, rerank=True, top_k=10, top_n=3)
2026-10-11 22:29:50 | INFO    | hybrid_retriever:retrieve:33 - Bắt đầu Hybrid Search: 'Số lượng ảnh lớp Implant trong tập test của bộ dữ liệu là bao nhiêu?' (dense_w=0.6, sparse_w=0.4)
2026-10-11 22:29:50 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Số lượng ảnh lớp Implant trong tập test của bộ dữ liệu là bao nhiêu?' (top_k=10)
2026-10-11 22:29:50 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-11 22:29:50 | INFO    | hybrid_retriever:retrieve:103 - Hoàn tất Hybrid Search. Trả về 10 chunks.
2026-10-11 22:29:50 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 10 chunks với câu hỏi: 'Số lượng ảnh lớp Implant trong tập test của bộ dữ liệu là bao nhiêu?' (top_n=3)
2026-10-11 22:29:54 | INFO   

## 🆔 q031: Số lượng ảnh lớp Implant trong tập test của bộ dữ liệu là bao nhiêu?

**🎯 Ground truth:** `104 ảnh.`

**❌ Model trả lời:** `<không có>`

---

### 📄 Chunk 1 | page=?
```
Về phân bố dữ liệu giữa các lớp, có thể thấy sự chênh lệch rất lớn. Lớp Normal chiếm ưu thế vượt trội với 17.400 ảnh trong tập huấn luyện, trong khi lớp Cavity chỉ có 576 ảnh và lớp Impacted Tooth chỉ có 428 ảnh. Tỉ lệ mất cân bằng giữa lớp nhiều nhất và ít nhất lên tới khoảng 30:1. Sự mất cân bằng này cũng thể hiện rõ ở trên tập valid và test, với lớp Normal có 1.941 ảnh (tập valid) và 1.241 ảnh (tập test), trong khi Cavity chỉ có 43 và 22 ảnh tương ứng.
```

### 📄 Chunk 2 | page=?
```
│ ├── Impacted Tooth/

│ ├── Implant/

│ ├── Normal/

│

│── test/

│ ├── Cavity/

│ ├── Fillings/

│ ├── Impacted Tooth/

│ ├── Implant/

│ ├── Normal/

Chi tiết tập dữ liệu:

*Bảng (thuộc: Tập dữ liệu)*
 | train | valid | test
--- | --- | --- | ---
Cavity | 576 | 43 | 22
Fillings | 5242 | 540 | 315
Impacted Tooth | 428 | 38 | 32
Implant | 1784 | 159 | 104
Normal | 17.4k | 1941 | 1241
Total | 25136 | 2812 | 1649

Bảng 1: Bảng mô tả chi tiết số lượng dữ liệu của dataset
```

### 📄 Chunk 3 | page=?
```
Bảng 1: Bảng mô tả chi tiết số lượng dữ liệu của dataset

Tập dữ liệu được tổ chức theo từng thư mục, mỗi thư mục đại diện cho một lớp tình trạng cần phân loại và chứa các ảnh X-quang tương ứng. Dữ liệu được chia thành ba tập huấn luyện (train), đánh giá (valid) và kiểm tra (test). Tổng số lượng ảnh trong tập huấn luyện là 25.136 ảnh, tập xác nhận là 2.812 ảnh và tập kiểm tra là 1.649 ảnh.
```


2026-10-11 22:29:54 | INFO    | retrieval_pipeline:retrieve:59 - Retrieval Pipeline cho query: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (hybrid=True, rerank=True, top_k=10, top_n=3)
2026-10-11 22:29:54 | INFO    | hybrid_retriever:retrieve:33 - Bắt đầu Hybrid Search: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (dense_w=0.6, sparse_w=0.4)
2026-10-11 22:29:54 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (top_k=10)
2026-10-11 22:29:54 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-11 22:29:54 | INFO    | hybrid_retriever:retrieve:103 - Hoàn tất Hybrid Search. Trả về 10 chunks.
2026-10-11 22:29:54 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 10 chunks với câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (top_n=3)
2026-10-11 22:29:58 | INFO    | reran

## 🆔 q032: Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?

**🎯 Ground truth:** `0.9597 (95.97%).`

**❌ Model trả lời:** `<không có>`

---

### 📄 Chunk 1 | page=?
```
Mô hình EfficientNetB0 cho thấy tốc độ hội tụ nhanh hơn DenseNet201, chỉ sau 10 epoch mà accuracy trên tập train đã đạt gần 90% nhưng val_loss vẫn còn khá cao. Ở epoch thứ 15 có thể thấy ReduceLROnPlateau đã kích hoạt để giảm tốc độ học xuống còn 1e-5 giúp val_loss bắt đầu ổn định và giảm sâu. Đến epoch 25 có thể thấy accuracy trên tập val đạt gần 90% và val_loss chỉ còn 0.3696 cho thấy mô hình đang được học tốt. Và kết quả cho thấy mô hình có val_loss tốt nhất tại eopch thứ 26.
```

### 📄 Chunk 2 | page=?
```
Kết quả thực nghiệm cho thấy kiến trúc DenseNet201 thể hiện sự ưu việt với độ chính xác (Accuracy) đạt 93% và giá trị mất mát (Loss) dừng ở mức 0.27. Chỉ số Loss thấp phản ánh sự hội tụ ổn định của mạng, cho thấy sai số giữa giá trị dự đoán và nhãn thực tế đã được tối thiểu hóa hiệu quả. Với độ chính xác 93%, DenseNet201 chứng minh khả năng trích xuất các đặc trưng hình thái phức tạp trên ảnh X-quang nha khoa là rất đáng tin cậy.
```

### 📄 Chunk 3 | page=?
```
Mô hình kết thúc tại epoch 46 do cơ chế EarlyStopping với patience là 20, và kết quả đạt được cho thấy mô hình tốt nhất tại epoch thứ 26 với accuracy trên tập test là hơn 90% và val loss chỉ còn 0.3375. Có thể thấy hiệu suất và độ hội tụ của EfficientNetB0 thấp hơn DenseNet201.
```


2026-10-11 22:29:58 | INFO    | retrieval_pipeline:retrieve:59 - Retrieval Pipeline cho query: 'Theo Bảng 6, lớp nào có chỉ số Precision cao nhất trên tập test?' (hybrid=True, rerank=True, top_k=10, top_n=3)
2026-10-11 22:29:58 | INFO    | hybrid_retriever:retrieve:33 - Bắt đầu Hybrid Search: 'Theo Bảng 6, lớp nào có chỉ số Precision cao nhất trên tập test?' (dense_w=0.6, sparse_w=0.4)
2026-10-11 22:29:58 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Theo Bảng 6, lớp nào có chỉ số Precision cao nhất trên tập test?' (top_k=10)
2026-10-11 22:29:58 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-11 22:29:58 | INFO    | hybrid_retriever:retrieve:103 - Hoàn tất Hybrid Search. Trả về 10 chunks.
2026-10-11 22:29:58 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 10 chunks với câu hỏi: 'Theo Bảng 6, lớp nào có chỉ số Precision cao nhất trên tập test?' (top_n=3)
2026-10-11 22:30:01 | INFO    | reranker:rera

...

## 🆔 q037: Dựa vào Bảng 4 và Bảng 5, hãy so sánh tốc độ hội tụ của DenseNet201 và EfficientNetB0.

**🎯 Ground truth:** `EfficientNetB0 hội tụ nhanh hơn ở giai đoạn đầu (chỉ sau 10 epoch đã đạt accuracy train ~90%), nhưng DenseNet201 đạt giá trị val_acc cao hơn ở các epoch cuối (92.78% so với 90.37% của EfficientNetB0) và có val_loss thấp hơn (0.2997 so với 0.3629).`

**❌ Model trả lời:** `<không có>`

---

### 📄 Chunk 1 | page=?
```
Trong khi đó, mô hình EfficientNetB0 đạt điểm tối ưu tại Epoch thứ 26 với độ chính xác trên tập kiểm thử là 91% và giá trị Validation Loss là 0.34. Khi đặt lên bàn cân so sánh, DenseNet201 cho thấy hiệu suất vượt trội hơn về cả khả năng phân loại lẫn tốc độ hội tụ. Sự khác biệt này bắt nguồn từ đặc thù kiến trúc: DenseNet201 với cấu trúc 201 lớp cùng cơ chế kết nối dày đặc (Dense Blocks) cho phép tái sử dụng đặc trưng xuyên suốt các tầng, giúp mô hình học được các chi tiết ở nhiều cấp độ phân giải khác nhau
```

### 📄 Chunk 2 | page=?
```
Mô hình EfficientNetB0 cho thấy tốc độ hội tụ nhanh hơn DenseNet201, chỉ sau 10 epoch mà accuracy trên tập train đã đạt gần 90% nhưng val_loss vẫn còn khá cao. Ở epoch thứ 15 có thể thấy ReduceLROnPlateau đã kích hoạt để giảm tốc độ học xuống còn 1e-5 giúp val_loss bắt đầu ổn định và giảm sâu. Đến epoch 25 có thể thấy accuracy trên tập val đạt gần 90% và val_loss chỉ còn 0.3696 cho thấy mô hình đang được học tốt. Và kết quả cho thấy mô hình có val_loss tốt nhất tại eopch thứ 26.
```

### 📄 Chunk 3 | page=?
```
Mô hình kết thúc tại epoch 46 do cơ chế EarlyStopping với patience là 20, và kết quả đạt được cho thấy mô hình tốt nhất tại epoch thứ 26 với accuracy trên tập test là hơn 90% và val loss chỉ còn 0.3375. Có thể thấy hiệu suất và độ hội tụ của EfficientNetB0 thấp hơn DenseNet201.
```


2026-10-11 22:30:04 | INFO    | retrieval_pipeline:retrieve:59 - Retrieval Pipeline cho query: 'Theo kết quả phân tích ở Chương 2, gói thành viên nào chiếm tỷ trọng lớn nhất với 40.6% trong tập dữ liệu Fitness?' (hybrid=True, rerank=True, top_k=10, top_n=3)
2026-10-11 22:30:04 | INFO    | hybrid_retriever:retrieve:33 - Bắt đầu Hybrid Search: 'Theo kết quả phân tích ở Chương 2, gói thành viên nào chiếm tỷ trọng lớn nhất với 40.6% trong tập dữ liệu Fitness?' (dense_w=0.6, sparse_w=0.4)
2026-10-11 22:30:04 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Theo kết quả phân tích ở Chương 2, gói thành viên nào chiếm tỷ trọng lớn nhất với 40.6% trong tập dữ liệu Fitness?' (top_k=10)
2026-10-11 22:30:04 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-11 22:30:04 | INFO    | hybrid_retriever:retrieve:103 - Hoàn tất Hybrid Search. Trả về 10 chunks.
2026-10-11 22:30:04 | INFO    | reranker:rerank:61 - Bắt đầu Reranking

## 🆔 q042: Theo kết quả phân tích ở Chương 2, gói thành viên nào chiếm tỷ trọng lớn nhất với 40.6% trong tập dữ liệu Fitness?

**🎯 Ground truth:** `Gói Premium với 40.6%.`

**❌ Model trả lời:** `<không có>`

---

### 📄 Chunk 1 | page=23
```
Biểu đồ thể hiện cơ cấu loại thẻ thành viên (Basic, Standard, Premium,
Elite) theo từng thành phố. Có sự khác biệt rõ rệt về tỷ lệ lựa chọn gói tập giữa
các địa phương. Gói Premium chiếm tỷ lệ cao nhất ở hầu hết các thành phố, đặc
biệt nổi bật tại San Jose (54%), Bakersfield (51%) và San Francisco (40%). Gói
Standard phổ biến nhất tại Fresno (33%), Oakland (31%) và Long Beach (34%),
nhưng lại thấp nhất tại San Jose (19%) và Bakersfield (19%). Gói Elite tuy có tỷ
```

### 📄 Chunk 2 | page=20
```
Biểu đồ cho thấy cơ cấu thành viên tương đối ổn định, trong đó hai hạng
thẻ (vàng) và (đỏ) luôn chiếm ưu thế với hơn 60% tỷ trọng ở
```

### 📄 Chunk 3 | page=12
```
## 1.2 MỤC TIÊU ĐỒ ÁN
Để đảm bảo tính khoa học, chuyên nghiệp và khẳng định chất lượng đầu ra
của dự án, nhóm chúng em đã xác định lộ trình triển khai cụ thể qua các giai đoạn
từ xử lý dữ liệu đến xây dựng mô hình thực tiễn. Mục tiêu trọng tâm là khai thác
tối đa giá trị từ 2 tập dữ liệu Fitness Membership và Customer Shopping
Behavior nhằm đưa ra các giải pháp tối ưu hóa trải nghiệm khách hàng và hỗ trợ
quyết định kinh doanh. Cụ thể, các mục tiêu chi tiết được thiết lập như sau:
```


2026-10-11 22:30:07 | INFO    | retrieval_pipeline:retrieve:59 - Retrieval Pipeline cho query: 'Quy tắc gán nhãn trong bài toán phân loại churn hội viên Fitness được thực hiện như thế nào?' (hybrid=True, rerank=True, top_k=10, top_n=3)
2026-10-11 22:30:07 | INFO    | hybrid_retriever:retrieve:33 - Bắt đầu Hybrid Search: 'Quy tắc gán nhãn trong bài toán phân loại churn hội viên Fitness được thực hiện như thế nào?' (dense_w=0.6, sparse_w=0.4)
2026-10-11 22:30:07 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Quy tắc gán nhãn trong bài toán phân loại churn hội viên Fitness được thực hiện như thế nào?' (top_k=10)
2026-10-11 22:30:07 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-11 22:30:08 | INFO    | hybrid_retriever:retrieve:103 - Hoàn tất Hybrid Search. Trả về 10 chunks.
2026-10-11 22:30:08 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 10 chunks với câu hỏi: 'Quy tắc gán nhãn trong bài toán phân 

## 🆔 q044: Quy tắc gán nhãn trong bài toán phân loại churn hội viên Fitness được thực hiện như thế nào?

**🎯 Ground truth:** `Lớp 0 (Đang dùng): Hội viên ghé thăm trong vòng dưới 21 ngày, đại diện cho nhóm đang hoạt động ổn định (35.3%). Lớp 1 (Có nguy cơ): Hội viên không đến từ 21 đến dưới 45 ngày, thuộc nhóm có nguy cơ rời bỏ cần chăm sóc (39.6%). Hội viên ngừng đến trên 45 ngày, được xác định là đã rời bỏ dịch vụ (25.1%).`

**❌ Model trả lời:** `<không có>`

---

### 📄 Chunk 1 | page=63
```
# A. Gán nhãn cho dữ liệu
Để đào sâu hơn tập dữ liệu, không chỉ phân tích và khai thác ở bài toán phân cụm
nhóm em còn phân tích khám phá sang bài toán phân loại. Trước hết, nhóm
chúng em đã thực hiện bước dán nhãn mục tiêu nhằm xác định chân dung khách
hàng tiềm năng. Vì tập dữ liệu gốc chưa có sẵn cột phân loại, nhóm đã xây dựng
một bộ quy tắc định tính dựa trên ba yếu tố cốt lõi: sự cam kết (Subscription),
kinh nghiệm mua sắm (Previous Purchases) và mức độ hài lòng (Review Rating).
```

### 📄 Chunk 2 | page=37
```
# A. Gán nhãn cho dữ liệu
Vì tập dữ liệu ban đầu không có cột nhãn trực tiếp cho biết khách hàng đã
rời bỏ dịch vụ hay chưa, chúng em thực hiện gán nhãn dựa trên hành vi thực tế
của khách hàng.
Quá trình gán nhãn được thực hiện thông qua việc tính toán khoảng thời
gian biến động giữa ngày ghi nhận cuối cùng trong hệ thống (ngày tham chiếu:
22/07/2025) và ngày ghé thăm phòng tập gần nhất của từng khách hàng
(last_visit_date). Dựa trên thuộc tính days_since_last_visit vừa được trích xuất,
```

### 📄 Chunk 3 | page=71
```
phản ánh hành vi khách hàng chính xác và kịp thời hơn đáng kể.
Về ứng dụng kinh doanh , bước phát triển quan trọng nhất là triển khai các
mô hình này thành hệ thống vận hành thực tế. Cụ thể, mô hình phân loại Churn
(Fitness) có thể được tích hợp vào hệ thống CRM để tự động kích hoạt quy trình
chăm sóc khách hàng theo ngưỡng rủi ro hàng tuần. Đối với bài toán phân loại
khách hàng tiềm năng (Shopping), mô hình có thể được nhúng vào pipeline
```
